## **Business rules**

In [0]:
import requests
import json
import os
import time
import logging
from datetime import datetime
from pyspark.sql.functions import lit, current_timestamp

# Configuração de Logs
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configurações do Catálogo
CATALOG = "workspace"
SCHEMA = "br_legislative"
VOLUME = "landing_zone"
ENTITY = "despesas"

LANDING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{ENTITY}"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.bronze_{ENTITY}"

def fetch_expenses_for_deputy(deputy_id: int) -> list:
    """Fetches all expenses for a given deputy handling API pagination."""
    all_expenses = []
    page = 1
    
    while True:
        url = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{deputy_id}/despesas"
        params = {
            "formato": "json",
            "pagina": page,
            "itens": 100, # Traz 100 registros por página para otimizar
            "ano": 2024   # Filtrando um ano específico para manter o escopo controlado
        }
        
        response = requests.get(url, params=params)
        
        if response.status_code != 200:
            logger.error(f"API Error for deputy {deputy_id} on page {page}: {response.status_code}")
            break
            
        data = response.json().get('dados', [])
        
        if not data:
            # Se a lista vier vazia, acabaram as páginas
            break
            
        all_expenses.extend(data)
        logger.info(f"Deputy {deputy_id} | Page {page} fetched ({len(data)} records)")
        
        # Controle de paginação e Rate Limit (pausa de meio segundo para não ser bloqueado)
        page += 1
        time.sleep(0.5) 
        
    return all_expenses

def ingest_expenses_to_bronze():
    """Main pipeline to extract expenses for deputies and load into Bronze."""
    logger.info("Starting expenses ingestion pipeline...")
    
    # 1. Pegar IDs dos deputados já ingeridos (Limitado a 3 para teste!)
    logger.info("Fetching deputy IDs from bronze_deputados...")
    deputies_df = spark.sql(f"SELECT id FROM {CATALOG}.{SCHEMA}.bronze_deputados LIMIT 3")
    deputy_ids = [row['id'] for row in deputies_df.collect()]
    
    dbutils.fs.mkdirs(LANDING_PATH)
    all_data_to_save = []
    
    # 2. Iterar sobre os deputados e buscar despesas
    for d_id in deputy_ids:
        expenses = fetch_expenses_for_deputy(d_id)
        if expenses:
            # Adicionar o ID do deputado no payload para garantir a rastreabilidade
            for exp in expenses:
                exp['id_deputado'] = d_id
            all_data_to_save.extend(expenses)
            
    if not all_data_to_save:
        logger.warning("No expenses found. Exiting.")
        return
        
    # 3. Salvar RAW JSON no Volume
    timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')
    file_name = f"{ENTITY}_{timestamp_str}.json"
    file_path = f"{LANDING_PATH}/{file_name}"
    
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(all_data_to_save, f, ensure_ascii=False)
    logger.info(f"Raw data saved to {file_path}")
    
    # 4. Criar DataFrame e salvar na Bronze
    df = spark.createDataFrame(all_data_to_save)
    df = df.withColumn("source_file", lit(file_path)) \
           .withColumn("ingestion_timestamp", current_timestamp())
           
    df.write.format("delta").mode("append").saveAsTable(TABLE_NAME)
    logger.info(f"Success! {len(all_data_to_save)} expense records appended to {TABLE_NAME}")


### **Execution Code:**

In [0]:
try:
    ingest_expenses_to_bronze()
except Exception as e:
    logger.error(f"Pipeline failed: {str(e)}")

In [0]:
%sql 
SELECT * FROM workspace.br_legislative.bronze_despesas LIMIT 10

In [0]:
%sql
SELECT count(*) FROM workspace.br_legislative.bronze_despesas